# Дообучение YOLO (stage 3) на Google Colab: MOCS + доп. датасеты техники

Версия ноутбука `finetune_mocs` под Colab + улучшения качества. Стартуем **не с MOCS-весов, а с уже дообученных**
(`phase2_full/weights/best.pt`), поэтому фазы 1–2 повторять не нужно.

**Перед запуском**
1. `Runtime → Change runtime type → GPU` (T4 достаточно; L4/A100 быстрее).
2. Положите на Google Drive `MyDrive/cv/weights/best.pt` (лучшие веса после фазы 2). Путь меняется в ячейке «Конфигурация».
3. `🔑 Secrets` (слева) → добавьте `KAGGLE_USERNAME`, `KAGGLE_KEY` и, если нужен roboflow-датасет, `ROBOFLOW_API_KEY`;
   включите **Notebook access** для каждого.
4. Запускайте ячейки сверху вниз. Данные лежат на **локальном диске Colab** (`/content`), на Drive пишутся только веса и результаты.

**Что добавлено по сравнению с исходным ноутбуком**
- групповой train/val split (аугментированные копии одного кадра не попадают в val);
- исправлен поиск папки `labels` в `convert_yolo_native` (ломался, если в пути к датасету есть слово `images`);
- аудит подготовленных данных (пустые лейблы, распределение классов);
- baseline-оценка старых весов **по каждому датасету и классу отдельно**;
- дополнение «недостающих» классов в доп. датасетах псевдо-разметкой текущей моделью;
- `train_stage3.txt`: подвыборка MOCS под лимиты Colab + oversampling редких классов (repeat factor sampling);
- stage 3: низкий LR, `close_mosaic`, чекпойнты на Drive, возможность `resume`;
- сравнение «до/после» по классам.

In [1]:
from pathlib import Path
for p in ["/content/mocs_yolo/images/train",
          "/content/finetune_data",
          "/content/drive/MyDrive/cv/runs_finetune/baseline_per_dataset.csv"]:
    print(("есть  " if Path(p).exists() else "НЕТ   ") + p)

НЕТ   /content/mocs_yolo/images/train
НЕТ   /content/finetune_data
НЕТ   /content/drive/MyDrive/cv/runs_finetune/baseline_per_dataset.csv


## 0. Установка зависимостей

In [2]:
!pip install -q -U ultralytics kagglehub pycocotools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.3 MB/s eta 0:00:00


## 0b. Colab: GPU, Google Drive, ключи

Ключи берутся из **Colab Secrets**. Если их нет — `kagglehub.login()` попросит ввести логин/ключ вручную.

In [3]:
import os, shutil
from pathlib import Path

import torch
assert torch.cuda.is_available(), "GPU не включён: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("Свободно на /content: %.0f ГБ" % (shutil.disk_usage("/content").free / 1e9))

from google.colab import drive, userdata
drive.mount("/content/drive")


def _secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


_ku, _kk = _secret("KAGGLE_USERNAME"), _secret("KAGGLE_KEY")
if _ku and _kk:
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = _ku, _kk
    print("Kaggle: ключи взяты из Colab Secrets")
else:
    import kagglehub
    kagglehub.login()   # интерактивный ввод

GPU: Tesla T4
Свободно на /content: 70 ГБ
Mounted at /content/drive
Kaggle: ключи взяты из Colab Secrets


## 1. Конфигурация

- `START_WEIGHTS` — веса **после фазы 2** (`phase2_full/weights/best.pt`). Порядок классов должен совпадать с `CANONICAL_CLASSES` (проверяется ниже).
- Данные (`mocs_yolo`, `finetune_data`) — на локальном диске Colab: чтение десятков тысяч мелких файлов с Drive очень медленное.
- Результаты обучения (`RUNS_DIR`) — на Drive, чтобы пережить обрыв сессии.
- Параметры stage 3 собраны в одном месте — подстраивайте под скорость первой эпохи.

In [4]:
import os
from pathlib import Path

RUNNING_ON_KAGGLE = False

# --- Drive (переживает перезапуск среды) ---
DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks")               # <-- поправьте при необходимости
START_WEIGHTS = DRIVE_ROOT / "weights" / "best.pt"           # веса ПОСЛЕ фазы 2
RUNS_DIR = DRIVE_ROOT / "runs_finetune"                      # сюда пишет Ultralytics (абсолютный путь!)
VIDEO_DIR = DRIVE_ROOT / "videos"                            # тестовые видео для инференса

# --- локальный диск Colab (исчезает при перезапуске среды) ---
MOCS_YOLO_DIR = Path("/content/mocs_yolo")
WORK_ROOT = Path("/content/finetune_data")
WORK_ROOT.mkdir(exist_ok=True, parents=True)
RUNS_DIR.mkdir(exist_ok=True, parents=True)

# --- параметры stage 3 ---
IMGSZ = 640          # 800/960 поможет на мелкой технике вдали, но кратно дороже по времени
BATCH = 16
WORKERS = 2          # в Colab всего 1-2 vCPU: больше воркеров не ускорит, а замедлит
EPOCHS = 15
MOCS_BUDGET = 12000  # сколько MOCS-картинок брать в эпоху (None = все ~37 тыс. -> не влезет в лимиты Colab)
RFS_T = 0.10         # порог repeat factor sampling: классы, встречающиеся в <10% картинок, oversample-им
MAX_REPEAT = 4
PSEUDO_CONF = 0.6    # порог уверенности для псевдо-разметки недостающих классов

# Канонические 13 классов — порядок ДОЛЖЕН совпадать с тем, что было при обучении на MOCS
CANONICAL_CLASSES = [
    "worker", "tower_crane", "hanging_hook", "vehicle_crane", "roller", "bulldozer",
    "excavator", "truck", "loader", "pump_truck", "concrete_mixer", "pile_driver", "other_vehicle",
]
CANON_IDX = {name: i for i, name in enumerate(CANONICAL_CLASSES)}
IMG_EXT = (".jpg", ".jpeg", ".png")

assert START_WEIGHTS.exists(), f"Не найдены веса: {START_WEIGHTS}. Загрузите best.pt на Drive или поправьте путь."
print("Стартовые веса:", START_WEIGHTS)

Стартовые веса: /content/drive/MyDrive/Colab Notebooks/weights/best.pt


## 2. Диск-дружелюбные хелперы

`link_or_copy` — ядро экономии места: символическая ссылка вместо копирования файла.
Если по каким-то причинам симлинки недоступны (редкость на Linux/Kaggle/Colab, актуально
в основном для Windows без прав администратора) — падаем обратно на обычное копирование.

In [5]:
import json, shutil, random, tempfile
import xml.etree.ElementTree as ET
import yaml


_SYMLINK_SUPPORTED = None


def _check_symlink_support():
    """Проверяем ОДИН РАЗ, поддерживаются ли symlink-и в этой среде — на Windows без
    включённого Developer Mode/прав администратора os.symlink тихо падает, и без
    этой проверки вы бы молча копировали гигабайты файлов на каждый вызов
    link_or_copy, думая, что диск экономится, а на деле — нет."""
    global _SYMLINK_SUPPORTED
    if _SYMLINK_SUPPORTED is None:
        test_dir = Path(tempfile.gettempdir()) / "symlink_capability_test"
        test_dir.mkdir(exist_ok=True)
        src, dst = test_dir / "s.txt", test_dir / "d.txt"
        src.write_text("x")
        try:
            os.symlink(src, dst)
            _SYMLINK_SUPPORTED = True
        except OSError:
            _SYMLINK_SUPPORTED = False
        shutil.rmtree(test_dir, ignore_errors=True)
        if not _SYMLINK_SUPPORTED:
            print("!! Символические ссылки недоступны в этой среде (на Windows нужен "
                  "включённый Developer Mode или запуск от администратора). "
                  "Файлы будут КОПИРОВАТЬСЯ вместо симлинков — это займёт больше места "
                  "на диске и больше времени.", flush=True)
    return _SYMLINK_SUPPORTED


def link_or_copy(src_path, dst_path):
    """Симлинк вместо копирования — почти нулевой расход диска.
    Если symlink-и недоступны в этой среде — сразу копируем, не тратя время
    на повторную неудачную попытку симлинка на каждом из тысяч файлов."""
    dst_path = Path(dst_path)
    if dst_path.exists() or dst_path.is_symlink():
        return
    if _check_symlink_support():
        try:
            os.symlink(Path(src_path).resolve(), dst_path)
            return
        except OSError:
            pass
    shutil.copy(src_path, dst_path)


def _looks_like_coco(json_path):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return isinstance(data, dict) and {"images", "annotations", "categories"} <= set(data.keys())
    except Exception:
        return False


def _find_yolo_split_dirs(root):
    """Ищет пары images/+labels/ как соседей (train/, valid/, val/, test/) -
    типичная YOLO-раскладка, даже без data.yaml/classes.txt (частый случай
    на переупакованных kaggle-зеркалах roboflow-датасетов)."""
    split_dirs = []
    for images_dir in Path(root).rglob("images"):
        if images_dir.is_dir():
            labels_dir = images_dir.parent / "labels"
            if labels_dir.is_dir():
                split_dirs.append(images_dir.parent)
    return split_dirs


def detect_annotation_format(dataset_root):
    """Определяет формат разметки:
    'yolo' - есть data.yaml с полем names;
    'yolo_nolabelmap' - есть images/+labels/, но yaml/classes.txt нет
        (нужно будет передать class_names_override вручную);
    'coco' - json с images/annotations/categories;
    'voc' - xml."""
    root = Path(dataset_root)

    for yf in list(root.rglob("*.yaml")) + list(root.rglob("*.yml")):
        try:
            content = yaml.safe_load(open(yf, encoding="utf-8"))
            if isinstance(content, dict) and "names" in content:
                return "yolo", yf
        except Exception:
            continue

    json_files = [f for f in root.rglob("*.json") if _looks_like_coco(f)]
    if json_files:
        return "coco", json_files

    xml_files = list(root.rglob("*.xml"))
    if xml_files:
        return "voc", xml_files

    split_dirs = _find_yolo_split_dirs(root)
    if split_dirs:
        return "yolo_nolabelmap", split_dirs

    raise RuntimeError(
        f"В {dataset_root} не нашёл ни YOLO data.yaml, ни COCO .json, ни VOC .xml, "
        "ни пар images/+labels/. Проверьте структуру датасета вручную (os.walk)."
    )


def read_yolo_native_names(yaml_path):
    content = yaml.safe_load(open(yaml_path, encoding="utf-8"))
    names = content["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names, key=lambda x: int(x))]
    return names


print("Хелперы загружены")

Хелперы загружены


## 3. Конвертеры в YOLO-формат (все используют `link_or_copy`, ничего не дублируют на диске)

In [6]:
def build_filename_index(root):
    """Строим индекс 'имя файла -> путь' ОДИН РАЗ, вместо пересканирования
    всего дерева датасета на каждое отдельное изображение (то, из-за чего
    конвертация больших датасетов вроде MOCS могла зависать без единого
    признака прогресса)."""
    index = {}
    for p in Path(root).rglob("*"):
        if p.is_file():
            index.setdefault(p.name, p)
    return index


def convert_coco_to_yolo(coco_json_path, images_search_root, out_images_dir, out_labels_dir,
                          progress_every=2000):
    out_images_dir = Path(out_images_dir); out_labels_dir = Path(out_labels_dir)
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    coco = json.load(open(coco_json_path, encoding="utf-8"))
    categories = sorted(coco["categories"], key=lambda c: c["id"])
    cat_id_to_idx = {c["id"]: i for i, c in enumerate(categories)}
    class_names = [c["name"] for c in categories]

    images_by_id = {img["id"]: img for img in coco["images"]}
    anns_by_image = {}
    for ann in coco["annotations"]:
        anns_by_image.setdefault(ann["image_id"], []).append(ann)

    print(f"  строю индекс файлов в {images_search_root}...", flush=True)
    filename_index = build_filename_index(images_search_root)
    print(f"  индекс готов: {len(filename_index)} файлов", flush=True)

    n = 0
    total = len(images_by_id)
    for image_id, img_info in images_by_id.items():
        file_name = img_info["file_name"]
        w, h = img_info["width"], img_info["height"]

        src = Path(images_search_root) / file_name
        if not src.exists():
            src = filename_index.get(Path(file_name).name)
            if src is None:
                continue

        dst = out_images_dir / Path(file_name).name
        link_or_copy(src, dst)

        lines = []
        for ann in anns_by_image.get(image_id, []):
            x, y, bw, bh = ann["bbox"]
            cx, cy, nw, nh = (x + bw / 2) / w, (y + bh / 2) / h, bw / w, bh / h
            lines.append(f"{cat_id_to_idx[ann['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        (out_labels_dir / (Path(file_name).stem + ".txt")).write_text("\n".join(lines))
        n += 1
        if n % progress_every == 0:
            print(f"    ...{n}/{total}", flush=True)

    print(f"  готово: {n}/{total}", flush=True)
    return class_names, n


def convert_voc_to_yolo(xml_files, images_search_root, out_images_dir, out_labels_dir,
                         class_names=None, progress_every=2000):
    """Если class_names не переданы - собираем список классов автоматически по всем <name> в xml
    (важно: для НОВЫХ датасетов НЕЛЬЗЯ подставлять MOCS_CLASSES, у них свои классы!)."""
    out_images_dir = Path(out_images_dir); out_labels_dir = Path(out_labels_dir)
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    if class_names is None:
        discovered = set()
        for xml_path in xml_files:
            root = ET.parse(xml_path).getroot()
            for obj in root.findall("object"):
                discovered.add(obj.find("name").text)
        class_names = sorted(discovered)

    class_to_idx = {name: i for i, name in enumerate(class_names)}

    print(f"  строю индекс файлов в {images_search_root}...", flush=True)
    filename_index = build_filename_index(images_search_root)
    print(f"  индекс готов: {len(filename_index)} файлов", flush=True)

    n = 0
    total = len(xml_files)
    for xml_path in xml_files:
        root = ET.parse(xml_path).getroot()
        filename = root.find("filename").text
        size = root.find("size")
        w, h = int(size.find("width").text), int(size.find("height").text)

        src = filename_index.get(Path(filename).name)
        if src is None:
            continue
        dst = out_images_dir / Path(filename).name
        link_or_copy(src, dst)

        lines = []
        for obj in root.findall("object"):
            name = obj.find("name").text
            if name not in class_to_idx:
                continue
            bbox = obj.find("bndbox")
            xmin, ymin = float(bbox.find("xmin").text), float(bbox.find("ymin").text)
            xmax, ymax = float(bbox.find("xmax").text), float(bbox.find("ymax").text)
            cx, cy = ((xmin + xmax) / 2) / w, ((ymin + ymax) / 2) / h
            nw, nh = (xmax - xmin) / w, (ymax - ymin) / h
            lines.append(f"{class_to_idx[name]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        (out_labels_dir / (Path(filename).stem + ".txt")).write_text("\n".join(lines))
        n += 1
        if n % progress_every == 0:
            print(f"    ...{n}/{total}", flush=True)

    print(f"  готово: {n}/{total}", flush=True)
    return class_names, n


def _images_to_labels(images_dir):
    """Заменяет ПОСЛЕДНИЙ компонент пути 'images' на 'labels'. Исходный вариант
    str(path).replace("images", "labels") менял ВСЕ вхождения - в том числе в имени датасета
    (например, .../construction-vehicle-images/...) - и тогда лейблы не находились,
    а картинки молча получали ПУСТУЮ разметку."""
    parts = list(Path(images_dir).parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i] == "images":
            parts[i] = "labels"
            return Path(*parts)
    return Path(images_dir).parent / "labels"


def convert_yolo_native(yaml_path, out_images_dir, out_labels_dir):
    """Датасет уже в YOLO-формате со своим data.yaml (частый случай для Roboflow-экспортов) -
    просто симлинкуем картинки и лейблы как есть, без перезаписи координат."""
    ds_root = Path(yaml_path).parent
    content = yaml.safe_load(open(yaml_path, encoding="utf-8"))
    class_names = read_yolo_native_names(yaml_path)

    out_images_dir = Path(out_images_dir); out_labels_dir = Path(out_labels_dir)
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)

    src_images_dirs = []
    for split_key in ("train", "val"):
        val = content.get(split_key)
        if val:
            p = (ds_root / val) if not os.path.isabs(val) else Path(val)
            if p.is_dir():
                src_images_dirs.append(p)
    if not src_images_dirs:
        src_images_dirs = list(ds_root.rglob("images"))

    n = 0
    for img_dir in src_images_dirs:
        lbl_dir = _images_to_labels(img_dir)
        for img_path in img_dir.glob("*"):
            if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            link_or_copy(img_path, out_images_dir / img_path.name)
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            if lbl_path.exists():
                link_or_copy(lbl_path, out_labels_dir / lbl_path.name)
            else:
                (out_labels_dir / (img_path.stem + ".txt")).write_text("")
            n += 1

    return class_names, n


def convert_yolo_splits(split_dirs, out_images_dir, out_labels_dir, progress_every=2000):
    """Датасет в YOLO-формате БЕЗ data.yaml/classes.txt - есть только пары images/+labels/
    (train/, valid/, test/...). Сливаем все сплиты в единый пул - дальше по нему
    проедется наш общий split_train_val 90/10. Имена классов сюда явно передаются
    отдельно через class_names_override в prepare_dataset, угадать их неоткуда."""
    out_images_dir = Path(out_images_dir); out_labels_dir = Path(out_labels_dir)
    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_labels_dir.mkdir(parents=True, exist_ok=True)
    total = sum(1 for d in split_dirs for p in (d / "images").glob("*")
                if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    print(f"  всего изображений по всем сплитам: {total}", flush=True)
    n = 0
    for split_dir in split_dirs:
        img_dir, lbl_dir = split_dir / "images", split_dir / "labels"
        for img_path in img_dir.glob("*"):
            if img_path.suffix.lower() not in (".jpg", ".jpeg", ".png"):
                continue
            dst_img = out_images_dir / f"{split_dir.name}__{img_path.name}"
            link_or_copy(img_path, dst_img)
            lbl_path = lbl_dir / (img_path.stem + ".txt")
            dst_lbl = out_labels_dir / (dst_img.stem + ".txt")
            if lbl_path.exists():
                link_or_copy(lbl_path, dst_lbl)
            else:
                dst_lbl.write_text("")
            n += 1
            if n % progress_every == 0:
                print(f"    ...{n}/{total}", flush=True)
    print(f"  готово: {n}/{total}", flush=True)

    return n


def print_class_distribution(labels_dir, class_names):
    """Печатает, сколько раз встречается каждый индекс - свериться глазами, что
    предполагаемый порядок классов реально совпадает с индексами внутри .txt."""
    from collections import Counter
    counts = Counter()
    for txt_path in Path(labels_dir).glob("*.txt"):
        for line in txt_path.read_text().splitlines():
            if line.strip():
                counts[int(line.split()[0])] += 1
    for i, name in enumerate(class_names):
        print(f"  {i:2d}: {name:<25s} {counts.get(i, 0)} инстансов")


def remap_and_filter_labels(src_labels_dir, dst_labels_dir, src_names, name_map):
    """Переписывает YOLO .txt: старые индексы -> канонические 13 классов (по ИМЕНИ класса).
    Классы без соответствия в name_map выбрасываются построчно (картинка не удаляется целиком)."""
    src_labels_dir = Path(src_labels_dir); dst_labels_dir = Path(dst_labels_dir)
    dst_labels_dir.mkdir(parents=True, exist_ok=True)

    idx_remap, dropped_classes = {}, []
    for i, name in enumerate(src_names):
        canon_name = name_map.get(name.strip().lower())
        if canon_name is None:
            dropped_classes.append(name)
            continue
        idx_remap[i] = CANON_IDX[canon_name]

    n_files = n_kept = n_dropped = 0
    for txt_path in src_labels_dir.glob("*.txt"):
        out_lines = []
        for line in txt_path.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            old_idx = int(parts[0])
            if old_idx in idx_remap:
                parts[0] = str(idx_remap[old_idx])
                out_lines.append(" ".join(parts))
                n_kept += 1
            else:
                n_dropped += 1
        (dst_labels_dir / txt_path.name).write_text("\n".join(out_lines))
        n_files += 1

    return {"files": n_files, "boxes_kept": n_kept, "boxes_dropped": n_dropped, "dropped_classes": dropped_classes}


def split_train_val(images_dir, labels_dir, out_root, val_fraction=0.1, seed=42,
                     move=False, progress_every=2000):
    """move=True - перемещаем файлы вместо копирования (мгновенно, не зависит от
    symlink-ов) - используйте только когда images_dir - временная папка, которую
    и так удалите сразу после (как raw_dir в prepare_dataset). Для MOCS-пересплита
    в существующей на месте папке move НЕ используйте - там источник и назначение
    пересекаются."""
    images_dir = Path(images_dir); labels_dir = Path(labels_dir); out_root = Path(out_root)
    images = [p for p in images_dir.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
    random.seed(seed)
    random.shuffle(images)
    split_idx = int(len(images) * (1 - val_fraction))
    splits = {"train": images[:split_idx], "val": images[split_idx:]}

    total = len(images)
    done = 0
    for split_name, imgs in splits.items():
        img_out = out_root / f"images/{split_name}"
        lbl_out = out_root / f"labels/{split_name}"
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        for img_path in imgs:
            lbl_path = labels_dir / (img_path.stem + ".txt")
            dst_img, dst_lbl = img_out / img_path.name, lbl_out / lbl_path.name
            if move:
                if img_path.resolve() != dst_img.resolve():
                    shutil.move(str(img_path), str(dst_img))
                if lbl_path.exists() and lbl_path.resolve() != dst_lbl.resolve():
                    shutil.move(str(lbl_path), str(dst_lbl))
            else:
                link_or_copy(img_path, dst_img)
                if lbl_path.exists():
                    link_or_copy(lbl_path, dst_lbl)
            done += 1
            if done % progress_every == 0:
                print(f"    ...{done}/{total}", flush=True)
    print(f"  train/val готово: {done}/{total}", flush=True)
    return {k: len(v) for k, v in splits.items()}


def prepare_dataset(dataset_root, name_map, out_dir, val_fraction=0.1, class_names_override=None):
    """Единая точка входа: скачанный датасет произвольного формата -> папка *_yolo/
    с картинками (симлинки) и лейблами, уже переписанными в канонические 13 классов.

    class_names_override - передайте явно, если у датасета нет data.yaml/classes.txt
    (формат 'yolo_nolabelmap') и вы знаете порядок классов из другого источника
    (например, со страницы датасета на Kaggle)."""
    out_dir = Path(out_dir)
    if (out_dir / "images/train").exists() and any((out_dir / "images/train").iterdir()):
        print(f"  {out_dir} уже подготовлен, пропускаю (удалите папку вручную, если нужно пересобрать)")
        return out_dir

    raw_dir = out_dir / "_raw"
    fmt, info = detect_annotation_format(dataset_root)
    print(f"  формат: {fmt}")

    if fmt == "yolo_nolabelmap":
        if class_names_override is None:
            raise RuntimeError(
                "Датасет в YOLO-формате без data.yaml/classes.txt - "
                "нужно передать class_names_override с известным порядком классов."
            )
        n = convert_yolo_splits(info, raw_dir / "images", raw_dir / "labels")
        class_names = class_names_override
        print("  распределение классов ДО ремаппинга (сверьте глазами!):")
        print_class_distribution(raw_dir / "labels", class_names)
    elif fmt == "yolo":
        class_names, n = convert_yolo_native(info, raw_dir / "images", raw_dir / "labels")
    elif fmt == "coco":
        train_json = info[0]
        class_names, n = convert_coco_to_yolo(train_json, dataset_root, raw_dir / "images", raw_dir / "labels")
    else:  # voc
        class_names, n = convert_voc_to_yolo(info, dataset_root, raw_dir / "images", raw_dir / "labels")

    print(f"  найдено классов: {class_names}")
    print(f"  сконвертировано изображений: {n}")

    remap_stats = remap_and_filter_labels(raw_dir / "labels", out_dir / "_remapped_labels", class_names, name_map)
    print(f"  ремаппинг: {remap_stats}")
    if remap_stats["dropped_classes"]:
        print(f"  !! классы без маппинга (проигнорированы): {remap_stats['dropped_classes']}")

    split_stats = split_train_val(raw_dir / "images", out_dir / "_remapped_labels", out_dir, val_fraction, move=True)
    print(f"  train/val: {split_stats}")
    shutil.rmtree(raw_dir, ignore_errors=True)
    shutil.rmtree(out_dir / "_remapped_labels", ignore_errors=True)
    return out_dir


print("Конвертеры загружены")

Конвертеры загружены


## 3b. Групповой train/val split

Исходный `split_train_val` перемешивает картинки случайно. У roboflow-датасетов в train лежат **аугментированные копии**
одного кадра (`имя_jpg.rf.<hash>.jpg`); при случайном 90/10 копии попадают и в train, и в val — метрики завышаются,
а `best.pt` выбирается по «подглядыванию». Здесь копии одного исходного кадра всегда уходят в одну сторону.
Для датасетов без такого именования поведение эквивалентно обычному случайному сплиту.

In [7]:
import re
from collections import defaultdict

_SPLIT_PREFIX = re.compile(r"^(train|valid|val|test)__")


def _group_key(stem):
    s = _SPLIT_PREFIX.sub("", stem)
    if ".rf." in s:                                   # roboflow: <имя>_jpg.rf.<hash>
        s = s.split(".rf.")[0]
        s = re.sub(r"_(jpg|jpeg|png)$", "", s, flags=re.I)
    return s


def split_train_val(images_dir, labels_dir, out_root, val_fraction=0.1, seed=42,
                     move=False, progress_every=2000):
    """Замена исходной функции (та же сигнатура), но split идёт по ГРУППАМ, а не по файлам."""
    images_dir = Path(images_dir); labels_dir = Path(labels_dir); out_root = Path(out_root)
    images = sorted(p for p in images_dir.glob("*") if p.suffix.lower() in IMG_EXT)

    groups = defaultdict(list)
    for p in images:
        groups[_group_key(p.stem)].append(p)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)

    val_target = max(1, int(len(images) * val_fraction)) if len(images) > 1 else 0
    val_keys, n_val = set(), 0
    for k in keys:
        if n_val >= val_target:
            break
        val_keys.add(k)
        n_val += len(groups[k])
    splits = {
        "train": [p for k in keys if k not in val_keys for p in groups[k]],
        "val": [p for k in keys if k in val_keys for p in groups[k]],
    }

    total, done = len(images), 0
    for split_name, imgs in splits.items():
        img_out = out_root / f"images/{split_name}"
        lbl_out = out_root / f"labels/{split_name}"
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        for img_path in imgs:
            lbl_path = labels_dir / (img_path.stem + ".txt")
            dst_img, dst_lbl = img_out / img_path.name, lbl_out / lbl_path.name
            if move:
                if img_path.resolve() != dst_img.resolve():
                    shutil.move(str(img_path), str(dst_img))
                if lbl_path.exists() and lbl_path.resolve() != dst_lbl.resolve():
                    shutil.move(str(lbl_path), str(dst_lbl))
            else:
                link_or_copy(img_path, dst_img)
                if lbl_path.exists():
                    link_or_copy(lbl_path, dst_lbl)
            done += 1
            if done % progress_every == 0:
                print(f"    ...{done}/{total}", flush=True)
    print(f"  train/val готово: {done}/{total} (групп: {len(keys)}, в val: {len(val_keys)})", flush=True)
    return {k: len(v) for k, v in splits.items()}

## 4. MOCS — только если ещё не сконвертирован

**Не пересобираем MOCS, если `mocs_yolo/` уже существует с прошлого запуска** — это самый большой
датасет, пересобирать его лишний раз незачем (и лишний расход диска, и время).

In [8]:
# Официальные названия классов в MOCS отличаются написанием от CANONICAL_CLASSES
# ("Static crane" против "tower_crane" и т.п.), хотя по смыслу это то же самое.
# Ремаппим ПО ИМЕНИ, как и остальные 4 датасета - так надёжнее, чем полагаться
# на то, что порядок категорий в исходном json всегда будет идентичен нашему.
MOCS_NAME_MAP = {
    "worker": "worker",
    "static crane": "tower_crane",
    "hanging head": "hanging_hook",
    "crane": "vehicle_crane",
    "roller": "roller",
    "bulldozer": "bulldozer",
    "excavator": "excavator",
    "truck": "truck",
    "loader": "loader",
    "pump truck": "pump_truck",
    "concrete mixer": "concrete_mixer",
    "pile driving": "pile_driver",
    "other vehicle": "other_vehicle",
}

if (MOCS_YOLO_DIR / "images/train").exists():
    print(f"MOCS уже сконвертирован в {MOCS_YOLO_DIR}, пропускаем")
else:
    if RUNNING_ON_KAGGLE:
        MOCS_DATASET_ROOT = "/kaggle/input/mocs-dataset"  # смонтирован, диск не расходует
    else:
        import kagglehub
        MOCS_DATASET_ROOT = kagglehub.dataset_download("xiaopan9802/mocs-dataset")

    print("MOCS путь:", MOCS_DATASET_ROOT)
    fmt, info = detect_annotation_format(MOCS_DATASET_ROOT)
    print("формат MOCS:", fmt)

    raw_dir = MOCS_YOLO_DIR / "_raw"

    if fmt == "coco":
        train_json = next((f for f in info if "train" in str(f).lower()), info[0])
        val_json = next((f for f in info if "val" in str(f).lower()), None)
        class_names, n_train = convert_coco_to_yolo(
            train_json, MOCS_DATASET_ROOT, raw_dir / "images/train", raw_dir / "labels/train"
        )
        print(f"MOCS train: {n_train} изображений")
        if val_json:
            convert_coco_to_yolo(val_json, MOCS_DATASET_ROOT, raw_dir / "images/val", raw_dir / "labels/val")
        else:
            print("Отдельного val.json нет - сплит 90/10 сделаем ниже из train")
    elif fmt == "voc":
        class_names, n_all = convert_voc_to_yolo(
            info, MOCS_DATASET_ROOT, raw_dir / "images/train", raw_dir / "labels/train", class_names=None
        )
    else:  # уже в YOLO
        class_names, n = convert_yolo_native(info, raw_dir / "images/train", raw_dir / "labels/train")

    print("Классы в исходной MOCS-разметке:", class_names)

    for split in ("train", "val"):
        src_labels = raw_dir / "labels" / split
        if not src_labels.exists():
            continue
        stats = remap_and_filter_labels(src_labels, MOCS_YOLO_DIR / "labels" / split, class_names, MOCS_NAME_MAP)
        print(f"  {split}: {stats}")
        if stats["dropped_classes"]:
            raise RuntimeError(
                f"В MOCS есть классы без соответствия в MOCS_NAME_MAP: {stats['dropped_classes']}. "
                "Для официального датасета таких быть не должно - проверьте название вручную."
            )
        img_out = MOCS_YOLO_DIR / "images" / split
        img_out.mkdir(parents=True, exist_ok=True)
        for img_path in (raw_dir / "images" / split).glob("*"):
            link_or_copy(img_path, img_out / img_path.name)

    val_dir = MOCS_YOLO_DIR / "images/val"
    if not val_dir.exists() or not any(val_dir.iterdir()):
        print("val отдельно не было - делаю сплит 90/10 из train")
        split_train_val(MOCS_YOLO_DIR / "images/train", MOCS_YOLO_DIR / "labels/train", MOCS_YOLO_DIR, val_fraction=0.1)

    shutil.rmtree(raw_dir, ignore_errors=True)
    class_names = CANONICAL_CLASSES
    print("MOCS приведён к канонической таксономии:", class_names)

100%|██████████| 8.96G/8.96G [02:14<00:00, 71.5MB/s]

Extracting files...


MOCS путь: /root/.cache/kagglehub/datasets/xiaopan9802/mocs-dataset/versions/1
формат MOCS: coco
  строю индекс файлов в /root/.cache/kagglehub/datasets/xiaopan9802/mocs-dataset/versions/1...
  индекс готов: 23407 файлов
    ...2000/19404
    ...4000/19404
    ...6000/19404
    ...8000/19404
    ...10000/19404
    ...12000/19404
    ...14000/19404
    ...16000/19404
    ...18000/19404
  готово: 19404/19404
MOCS train: 19404 изображений
  строю индекс файлов в /root/.cache/kagglehub/datasets/xiaopan9802/mocs-dataset/versions/1...
  индекс готов: 23407 файлов
    ...2000/4000
    ...4000/4000
  готово: 4000/4000
Классы в исходной MOCS-разметке: ['Worker', 'Static crane', 'Hanging head', 'Crane', 'Roller', 'Bulldozer', 'Excavator', 'Truck', 'Loader', 'Pump truck', 'Concrete mixer', 'Pile driving', 'Other vehicle']
  train: {'files': 19404, 'boxes_kept': 116945, 'boxes_dropped': 0, 'dropped_classes': []}
  val: {'files': 4000, 'boxes_kept': 18965, 'boxes_dropped': 0, 'dropped_classes': []}

## 5. Карты соответствия классов для новых датасетов

Ключ — имя класса в исходном датасете (в нижнем регистре, как оно там называется), значение — один
из `CANONICAL_CLASSES`. Классы, которых нет в словаре (например, PPE-классы у roboflow-датасета —
`helmet`, `vest`, `no-helmet` и т.п., они не про технику), автоматически выбрасываются в `remap_and_filter_labels`.

In [9]:
NAME_MAP_XYZYXZZXY = {
    "dump truck": "truck",
    "excavator": "excavator",
    "motor grader": "other_vehicle",
    "roller": "roller",
    "crane manipulator": "vehicle_crane",
    "gazelle": "other_vehicle",
    "forklift standart": "other_vehicle",
    "bucket loader big": "loader",
    "mixer": "concrete_mixer",
    "tanker": "other_vehicle",
    "bulldozer": "bulldozer",
    "cleaning equipment": "other_vehicle",
    "truck": "truck",
    "trailer": "other_vehicle",
    "forklift giraffe": "other_vehicle",
    "bucket loader standart": "loader",
    "autocran": "vehicle_crane",
}

NAME_MAP_KARTAVIYCHERT = {
    "crane": "vehicle_crane",
    "excavator": "excavator",
    "truck": "truck",
    "tractor": "other_vehicle",
    "other": "other_vehicle",
}

NAME_MAP_DATACLUSTERLABS = {
    "truck": "truck",
    "tractor": "other_vehicle",
}

NAME_MAP_ROBOFLOW_JEJUNG = {
    "crane": "vehicle_crane",
    "pump-truck": "pump_truck",
    "roller": "roller",
    "excavator-crane": "excavator",   # составной класс, ближе всего к excavator - стоит перепроверить глазами
    "concrete truck": "concrete_mixer",
    "aerial work truck": "other_vehicle",
    "engineer vehicle": "other_vehicle",
    "generator": "other_vehicle",
    "trailer": "other_vehicle",
    "shovel": "excavator",
    "smalldigger": "excavator",
    # helmet, vest, cone, no-helmet, no-vest, safety shoes, no safety shoes, PA-helmet, PA-vest,
    # PanAsia engineer, HondaSUV, motorcycle, sub-helmet - намеренно не мапим, это не техника
}

print("Карты соответствия готовы")

Карты соответствия готовы


## 6. Скачивание и подготовка новых датасетов

Каждый датасет получает свою отдельную папку в `finetune_data/` — они не сливаются друг с другом
и тем более не сливаются с `mocs_yolo/`. `prepare_dataset()` сама определяет формат разметки,
конвертирует, ремаппит классы и делает train/val split (90/10).

Заполните способ скачивания под то, что у вас реально работает (`kagglehub` для kaggle-датасетов,
Roboflow SDK для roboflow — там нужен свой API-ключ, впишите его в `ROBOFLOW_API_KEY`).

На Colab каждая новая сессия начинается с пустого диска, поэтому этот раздел (скачивание + конвертация) выполняется заново; веса и результаты обучения при этом сохраняются на Drive.

In [10]:
import kagglehub

# У этого датасета нет data.yaml/classes.txt в раздаче - порядок классов
# берём со страницы Kaggle Data Explorer (0: Dump truck, 1: Excavator, ...).
# После конвертации ОБЯЗАТЕЛЬНО проверьте вывод print_class_distribution -
# если распределение выглядит неправдоподобно (например, у редкого класса
# тысячи инстансов), значит реальный порядок в файлах другой.
XYZYXZZXY_CLASS_NAMES = [
    "Dump truck", "Excavator", "Motor grader", "Roller", "Crane manipulator", "Gazelle",
    "Forklift Standart", "Bucket loader Big", "Mixer", "Tanker", "Bulldozer",
    "Cleaning equipment", "Truck", "Trailer", "Forklift Giraffe", "Bucket loader Standart", "Autocran",
]

datasets_to_prepare = [
    ("xyzyxzzxy/construction-equipment", NAME_MAP_XYZYXZZXY, "extra_xyzyxzzxy", XYZYXZZXY_CLASS_NAMES),
    ("kartaviychert/arh-df", NAME_MAP_KARTAVIYCHERT, "extra_kartaviychert", None),
    ("dataclusterlabs/construction-vehicle-images", NAME_MAP_DATACLUSTERLABS, "extra_dataclusterlabs", None),
]

prepared_dirs = []
for kaggle_slug, name_map, out_name, class_override in datasets_to_prepare:
    print(f"\n=== {kaggle_slug} ===")
    out_dir_check = WORK_ROOT / out_name
    if (out_dir_check / "images/train").exists() and any((out_dir_check / "images/train").iterdir()):
        print(f"  {out_dir_check} уже подготовлен, пропускаю (без скачивания)")
        prepared_dirs.append(out_dir_check)
        continue
    raw_root = kagglehub.dataset_download(kaggle_slug)
    out_dir = prepare_dataset(raw_root, name_map, WORK_ROOT / out_name, class_names_override=class_override)
    prepared_dirs.append(out_dir)


=== xyzyxzzxy/construction-equipment ===
Using Colab cache for faster access to the 'construction-equipment' dataset.
  формат: yolo_nolabelmap
  всего изображений по всем сплитам: 19982
    ...2000/19982
    ...4000/19982
    ...6000/19982
    ...8000/19982
    ...10000/19982
    ...12000/19982
    ...14000/19982
    ...16000/19982
    ...18000/19982
  готово: 19982/19982
  распределение классов ДО ремаппинга (сверьте глазами!):
   0: Dump truck                7048 инстансов
   1: Excavator                 8283 инстансов
   2: Motor grader              667 инстансов
   3: Roller                    21 инстансов
   4: Crane manipulator         847 инстансов
   5: Gazelle                   4883 инстансов
   6: Forklift Standart         971 инстансов
   7: Bucket loader Big         2454 инстансов
   8: Mixer                     1224 инстансов
   9: Tanker                    427 инстансов
  10: Bulldozer                 1 инстансов
  11: Cleaning equipment        865 инстансов
  12: Truck

100%|██████████| 407M/407M [00:03<00:00, 112MB/s] 

Extracting files...


  формат: yolo
  найдено классов: ['crane', 'excavator', 'other', 'tractor', 'truck']
  сконвертировано изображений: 318
  ремаппинг: {'files': 318, 'boxes_kept': 3752, 'boxes_dropped': 0, 'dropped_classes': []}
  train/val готово: 318/318 (групп: 318, в val: 31)
  train/val: {'train': 287, 'val': 31}

=== dataclusterlabs/construction-vehicle-images ===


100%|██████████| 1.06G/1.06G [00:13<00:00, 81.5MB/s]

Extracting files...


  формат: voc
  строю индекс файлов в /root/.cache/kagglehub/datasets/dataclusterlabs/construction-vehicle-images/versions/2...
  индекс готов: 652 файлов
  готово: 326/326
  найдено классов: ['tractor', 'truck']
  сконвертировано изображений: 326
  ремаппинг: {'files': 326, 'boxes_kept': 427, 'boxes_dropped': 0, 'dropped_classes': []}
  train/val готово: 326/326 (групп: 326, в val: 32)
  train/val: {'train': 294, 'val': 32}


### Roboflow (необязательно)

Ключ берётся из Colab Secrets (`ROBOFLOW_API_KEY`). Нет ключа или пакет не установился — ячейка пропускается, остальное работает без него.

In [ ]:
ROBOFLOW_API_KEY = _secret("ROBOFLOW_API_KEY")
_rf_out = WORK_ROOT / "extra_roboflow_jejung"

if _rf_out.exists() and any((_rf_out / "images/train").glob("*")):
    print("roboflow-датасет уже подготовлен, пропускаю")
    if _rf_out not in prepared_dirs:
        prepared_dirs.append(_rf_out)
elif not ROBOFLOW_API_KEY:
    print("ROBOFLOW_API_KEY не задан в Colab Secrets - roboflow-датасет пропущен")
else:
    try:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "roboflow"], check=True)
        from roboflow import Roboflow

        rf = Roboflow(api_key=ROBOFLOW_API_KEY)
        project = rf.workspace("jejung").project("construction-machinery-tbosw")
        version = project.version(1)
        roboflow_raw_dir = version.download("yolov8").location

        out_dir = prepare_dataset(roboflow_raw_dir, NAME_MAP_ROBOFLOW_JEJUNG, _rf_out)
        prepared_dirs.append(out_dir)
    except Exception as e:
        print(f"Roboflow-датасет пропущен: {e}")
        print("Остальной пайплайн продолжит работать и без него.")

ROBOFLOW_API_KEY не задан в Colab Secrets - roboflow-датасет пропущен


## 7. Аудит подготовленных данных

Быстрая проверка до долгого обучения: доля пустых лейблов и распределение классов по датасетам.
Если у extras почти все лейблы пустые — что-то сломалось при конвертации/ремаппинге, обучаться на этом нельзя
(модель будет учиться, что на этих картинках техники нет).

In [11]:
import pandas as pd
from collections import Counter

DATASETS = {"mocs": MOCS_YOLO_DIR, **{d.name: d for d in prepared_dirs}}


def audit_dataset(name, ds_dir):
    rows = []
    for split in ("train", "val"):
        img_dir, lbl_dir = ds_dir / "images" / split, ds_dir / "labels" / split
        imgs = [p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXT] if img_dir.exists() else []
        no_file = empty = 0
        counts = Counter()
        for p in imgs:
            lp = lbl_dir / (p.stem + ".txt")
            if not lp.exists():
                no_file += 1
                continue
            lines = [l for l in lp.read_text().splitlines() if l.strip()]
            if not lines:
                empty += 1
            for l in lines:
                counts[int(l.split()[0])] += 1
        row = {"dataset": name, "split": split, "images": len(imgs), "no_label_file": no_file,
               "empty_%": round(100 * empty / max(1, len(imgs)), 1)}
        row.update({CANONICAL_CLASSES[i]: counts.get(i, 0) for i in range(len(CANONICAL_CLASSES))})
        rows.append(row)
    return rows


audit = pd.DataFrame([r for n, d in DATASETS.items() for r in audit_dataset(n, d)])
pd.set_option("display.width", 250, "display.max_columns", 30)
display(audit)

for _, r in audit.iterrows():
    if r["images"] and r["empty_%"] > 80:
        print(f"!! {r['dataset']}/{r['split']}: {r['empty_%']}% пустых лейблов - проверьте конвертацию/ремаппинг!")

,dataset,split,images,no_label_file,empty_%,worker,tower_crane,hanging_hook,vehicle_crane,roller,bulldozer,excavator,truck,loader,pump_truck,concrete_mixer,pile_driver,other_vehicle
0,mocs,train,19404,0,0.0,77770,7560,2734,2629,844,961,9435,5007,873,1443,1508,1051,5130
1,mocs,val,4000,0,0.0,11658,972,307,362,171,205,2622,1094,170,241,179,160,824
2,extra_xyzyxzzxy,train,17984,0,13.4,0,0,0,2365,17,1,7474,7502,4389,0,1113,0,9354
3,extra_xyzyxzzxy,val,1998,0,14.2,0,0,0,273,4,0,809,851,468,0,111,0,995
4,extra_kartaviychert,train,287,0,0.0,0,0,0,668,0,0,717,990,0,0,0,0,1015
5,extra_kartaviychert,val,31,0,0.0,0,0,0,71,0,0,73,117,0,0,0,0,101
6,extra_dataclusterlabs,train,294,0,2.4,0,0,0,0,0,0,0,275,0,0,0,0,109
7,extra_dataclusterlabs,val,32,0,0.0,0,0,0,0,0,0,0,33,0,0,0,0,10


## 8. Baseline: оценка текущих весов по датасетам и классам

Общий mAP на смеси датасетов скрывает, где модель слаба. Здесь `val` прогоняется **отдельно по каждому датасету**.
Для extras оцениваются только классы, которые в этом датасете реально размечены (`covered`): AP считается независимо по классам,
поэтому «неразмеченные» классы просто не смешиваются в метрику.

In [12]:
import pandas as pd, torch
from pathlib import Path
from ultralytics import YOLO

SEARCH_ROOT = DRIVE_ROOT      # если файлов нет, попробуйте Path("/content/drive/MyDrive")
COCO_HEAD = ["person", "bicycle", "car"]

rows = []
for p in sorted(SEARCH_ROOT.rglob("*.pt")):
    row = {"file": str(p.relative_to(SEARCH_ROOT)), "MB": round(p.stat().st_size / 1e6, 1)}
    try:
        m = YOLO(str(p))
        names = [m.names[i] for i in range(len(m.names))]
        row["nc"] = len(names)
        if names == CANONICAL_CLASSES:
            row["вердикт"] = "ПОДХОДИТ (13 классов, порядок совпадает)"
        elif names[:3] == COCO_HEAD:
            row["вердикт"] = "COCO-предобученные веса, не ваши"
        else:
            row["вердикт"] = f"другой набор классов: {names[:4]}..."
        ckpt = torch.load(p, map_location="cpu", weights_only=False)
        ta = ckpt.get("train_args") or {}
        row["data"] = str(ta.get("data", "-"))[-40:]
        row["epochs"] = ta.get("epochs", "-")
        row["date"] = str(ckpt.get("date", "-"))[:19]
    except Exception as e:
        row["вердикт"] = f"не читается: {type(e).__name__}"
    rows.append(row)

pd.set_option("display.width", 250, "display.max_colwidth", 60)
if rows:
    display(pd.DataFrame(rows))
else:
    print(f"В {SEARCH_ROOT} нет ни одного .pt файла")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


,file,MB,вердикт,nc,data,epochs,date
0,data/construction_phase_checkpoints/best.pt,43.3,не читается: TypeError,NaN,NaN,NaN,NaN
1,data/construction_phase_checkpoints/checkpoint_epoch_030.pt,43.3,не читается: TypeError,NaN,NaN,NaN,NaN
2,runs_finetune/stage3/weights/best.pt,207.4,"ПОДХОДИТ (13 классов, порядок совпадает)",13.0,/content/finetune_data/data_stage3.yaml,15.0,2026-09-22T12:09:12
3,runs_finetune/stage3/weights/last.pt,207.4,"ПОДХОДИТ (13 классов, порядок совпадает)",13.0,/content/finetune_data/data_stage3.yaml,15.0,2026-09-22T12:09:12
4,weights/best.pt,155.5,"ПОДХОДИТ (13 классов, порядок совпадает)",13.0,finetune_data\merged_data.yaml,20.0,2026-09-17T02:46:57
5,weights/yolo26n.pt,5.5,"COCO-предобученные веса, не ваши",80.0,ytics/ultralytics/cfg/datasets/coco.yaml,245.0,2025-12-15T06:36:05


In [13]:
START_WEIGHTS = DRIVE_ROOT / "weights" / "best.pt"   # <- ваше имя файла

assert START_WEIGHTS.exists(), f"Нет файла {START_WEIGHTS}"
m = YOLO(str(START_WEIGHTS))
names = [m.names[i] for i in range(len(m.names))]
assert names == CANONICAL_CLASSES, names
print("OK, веса подходят:", START_WEIGHTS)

OK, веса подходят: /content/drive/MyDrive/Colab Notebooks/weights/best.pt


In [14]:
from ultralytics import YOLO
import yaml

EXTRA_NAME_MAPS = {
    "extra_xyzyxzzxy": NAME_MAP_XYZYXZZXY,
    "extra_kartaviychert": NAME_MAP_KARTAVIYCHERT,
    "extra_dataclusterlabs": NAME_MAP_DATACLUSTERLABS,
    "extra_roboflow_jejung": NAME_MAP_ROBOFLOW_JEJUNG,
}
COVERED = {name: sorted(set(m.values())) for name, m in EXTRA_NAME_MAPS.items()}   # какие классы размечены в датасете


def check_class_order(model):
    names = [model.names[i] for i in range(len(model.names))]
    assert names == CANONICAL_CLASSES, f"Порядок классов в весах не совпадает с CANONICAL_CLASSES:\n{names}"


check_class_order(YOLO(str(START_WEIGHTS)))
print("Порядок классов в весах совпадает с CANONICAL_CLASSES")


def eval_per_dataset(weights, imgsz=IMGSZ, batch=BATCH):
    model = YOLO(str(weights))
    rows = []
    for ds_name, ds_dir in DATASETS.items():
        val_dir = ds_dir / "images/val"
        if not val_dir.exists() or not any(val_dir.iterdir()):
            continue
        yml = WORK_ROOT / f"_val_{ds_name}.yaml"
        yml.write_text(yaml.safe_dump({
            "train": str(val_dir), "val": str(val_dir),
            "names": {i: n for i, n in enumerate(CANONICAL_CLASSES)},
        }))
        m = model.val(data=str(yml), imgsz=imgsz, batch=batch, conf=0.001, iou=0.6, plots=False,
                      verbose=False, project="/content/_val_tmp", name=ds_name, exist_ok=True)
        keep = COVERED.get(ds_name)     # None = MOCS, все 13 классов
        for ci, a50, a in zip(m.box.ap_class_index, m.box.ap50, m.box.ap):
            cname = CANONICAL_CLASSES[int(ci)]
            if keep is None or cname in keep:
                rows.append({"dataset": ds_name, "class": cname, "AP50": float(a50), "AP50-95": float(a)})
    return pd.DataFrame(rows)


def compare_tables(**named):
    """compare_tables(before=df1, after=df2) -> таблица AP50-95 по (dataset, class) + дельта"""
    merged = None
    for tag, df in named.items():
        d = df.set_index(["dataset", "class"])[["AP50", "AP50-95"]].add_suffix(f" [{tag}]")
        merged = d if merged is None else merged.join(d, how="outer")
    tags = list(named)
    if len(tags) == 2:
        merged["Δ AP50-95"] = merged[f"AP50-95 [{tags[1]}]"] - merged[f"AP50-95 [{tags[0]}]"]
    return merged.round(3)


baseline_df = pd.read_csv(RUNS_DIR / "baseline_per_dataset.csv")
display(baseline_df.pivot(index="class", columns="dataset", values="AP50-95").round(3))

Порядок классов в весах совпадает с CANONICAL_CLASSES


dataset,extra_dataclusterlabs,extra_kartaviychert,extra_xyzyxzzxy,mocs
class,,,,
bulldozer,NaN,NaN,NaN,0.673
concrete_mixer,NaN,NaN,0.774,0.650
excavator,NaN,0.567,0.856,0.768
hanging_hook,NaN,NaN,NaN,0.327
loader,NaN,NaN,0.845,0.597
other_vehicle,0.635,0.364,0.754,0.400
pile_driver,NaN,NaN,NaN,0.429
pump_truck,NaN,NaN,NaN,0.688
roller,NaN,NaN,0.314,0.744


## 9. Псевдо-разметка «недостающих» классов в доп. датасетах

**Проблема.** Например, `dataclusterlabs` размечает только `truck`/`tractor`, а в roboflow-датасете `worker` не размечен вообще
(`helmet`, `vest`… выброшены при ремаппинге). Если на такой картинке стоит рабочий, то для модели это **явный фон** —
и обучение на этих данных «учит» её не замечать людей и краны. На MOCS-классах, которых нет в датасете, это даёт просадку recall.

**Решение.** Для каждого extra-датасета берём классы, которых в его `name_map` нет, и дописываем в **train**-лейблы
детекции текущей модели с высокой уверенностью (`PSEUDO_CONF`), пропуская боксы, пересекающиеся с уже размеченными.
Val не трогаем, чтобы метрики не стали кругом. Оригинальные лейблы сохраняются в `_label_backups/`, ячейку можно
безопасно перезапускать (лейблы каждый раз восстанавливаются из бэкапа).

Ограничение: это самоподтверждение (модель размечает сама себя). Оно нужно не для «обучения новому», а чтобы **не наказывать**
модель за то, что она правильно видит и просто не размечено в данном датасете.

In [15]:
import psutil, numpy as np
from PIL import Image

vm = psutil.virtual_memory()
print(f"ОЗУ: процесс {psutil.Process().memory_info().rss/1e9:.1f} ГБ, "
      f"занято в системе {vm.used/1e9:.1f} из {vm.total/1e9:.1f} ГБ")

for d in prepared_dirs:
    mp, bad = [], 0
    for p in (d / "images/train").glob("*"):
        try:
            with Image.open(p) as im:              # читается только заголовок, картинка не декодируется
                mp.append(im.size[0] * im.size[1] / 1e6)
        except Exception:
            bad += 1
    mp = np.array(mp)
    if len(mp):
        print(f"{d.name}: {len(mp)} картинок | мегапикселей: медиана {np.median(mp):.1f}, "
              f"макс {mp.max():.0f} | больше 20 МП: {(mp > 20).sum()} | нечитаемых/гигантских: {bad}")

ОЗУ: процесс 1.4 ГБ, занято в системе 2.6 из 13.6 ГБ
extra_xyzyxzzxy: 17984 картинок | мегапикселей: медиана 2.1, макс 4 | больше 20 МП: 0 | нечитаемых/гигантских: 0
extra_kartaviychert: 287 картинок | мегапикселей: медиана 3.7, макс 4 | больше 20 МП: 0 | нечитаемых/гигантских: 0
extra_dataclusterlabs: 294 картинок | мегапикселей: медиана 12.0, макс 24 | больше 20 МП: 5 | нечитаемых/гигантских: 0


In [16]:
import gc, psutil
import numpy as np
from PIL import Image, ImageOps

MAX_SIDE = 1280                                          # длинная сторона при чтении; боксы нормированные -> от этого не зависят
RAM_LIMIT_GB = 0.7 * psutil.virtual_memory().total / 1e9  # при превышении остановимся САМИ, не дожидаясь падения ядра


def _rss_gb():
    return psutil.Process().memory_info().rss / 1e9


def _read_small(path, max_side=MAX_SIDE):
    """Читает картинку сразу в уменьшенном виде: для JPEG PIL декодирует с масштабом 1/2, 1/4, 1/8,
    полный кадр в память не разворачивается. Возвращает BGR-массив, как ожидает Ultralytics."""
    with Image.open(path) as im:
        im.draft("RGB", (max_side, max_side))            # ничего не делает для не-JPEG
        im = ImageOps.exif_transpose(im)                 # ориентация как у cv2.imread при обучении
        im = im.convert("RGB")
        if max(im.size) > max_side:
            im.thumbnail((max_side, max_side))
        return np.ascontiguousarray(np.asarray(im)[..., ::-1])


def _iou_max(box, boxes):
    """box: (4,), boxes: (m,4), все xyxy в нормированных координатах; возвращает max IoU."""
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    a = (box[2] - box[0]) * (box[3] - box[1])
    b = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return float((inter / (a + b - inter + 1e-9)).max())


def add_pseudo_labels(model, ds_dir, missing_idx, conf=PSEUDO_CONF, iou_skip=0.3,
                      imgsz=IMGSZ, batch=4, backup_root=None, limit=None):
    """Картинки читаем и передаём в модель САМИ, по batch штук: в памяти одновременно только они.
    Никаких генераторов/списков путей Ultralytics, поведение по памяти полностью под контролем."""
    ds_dir = Path(ds_dir)
    img_dir, lbl_dir = ds_dir / "images/train", ds_dir / "labels/train"
    backup_root = Path(backup_root or (WORK_ROOT / "_label_backups"))
    bk = backup_root / ds_dir.name
    if not bk.exists():
        shutil.copytree(lbl_dir, bk)                     # первый запуск: сохраняем оригинал
    else:
        for f in bk.glob("*.txt"):                       # повторный запуск: стартуем с оригинала
            shutil.copy(f, lbl_dir / f.name)

    files = sorted(p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXT)
    if limit:
        files = files[:limit]
    added, touched, skipped = Counter(), 0, []

    for start in range(0, len(files), batch):
        chunk, imgs, ok = files[start:start + batch], [], []
        for p in chunk:
            try:
                imgs.append(_read_small(p)); ok.append(p)
            except Exception as e:                       # битая/гигантская картинка - пропускаем
                skipped.append((p.name, type(e).__name__))
        if imgs:
            results = model.predict(source=imgs, conf=conf, imgsz=imgsz,
                                    classes=sorted(missing_idx), verbose=False)
            for p, r in zip(ok, results):
                if r.boxes is None or len(r.boxes) == 0:
                    continue
                cls = r.boxes.cls.cpu().numpy().astype(int)
                xyxyn = r.boxes.xyxyn.cpu().numpy()
                xywhn = r.boxes.xywhn.cpu().numpy()
                lp = lbl_dir / (p.stem + ".txt")

                gt_lines, gt_boxes = [], []
                if lp.exists():
                    for line in lp.read_text().splitlines():
                        q = line.split()
                        if len(q) >= 5:
                            gt_lines.append(line.strip())
                            cx, cy, w, h = map(float, q[1:5])
                            gt_boxes.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
                gt = np.array(gt_boxes, dtype=float).reshape(-1, 4)

                new = []
                for c, bb, xywh in zip(cls, xyxyn, xywhn):
                    if len(gt) and _iou_max(bb, gt) > iou_skip:
                        continue                         # этот объект уже размечен (любым классом)
                    new.append(f"{int(c)} {xywh[0]:.6f} {xywh[1]:.6f} {xywh[2]:.6f} {xywh[3]:.6f}")
                    added[int(c)] += 1
                if new:
                    lp.write_text("\n".join(gt_lines + new))
                    touched += 1
            del results
        del imgs

        n_done = start + len(chunk)
        rss = _rss_gb()
        if rss > RAM_LIMIT_GB:                           # сторожок: лучше понятная ошибка, чем падение ядра
            raise MemoryError(f"ОЗУ процесса {rss:.1f} ГБ > лимита {RAM_LIMIT_GB:.1f} ГБ. "
                              f"Остановлено на {chunk[-1].name} ({n_done}/{len(files)}).")
        if (start // batch) % 100 == 0:
            gc.collect()
            print(f"    ...{n_done}/{len(files)}   ОЗУ процесса: {rss:.1f} ГБ", flush=True)

    if skipped:
        print(f"  пропущено нечитаемых картинок: {len(skipped)}, например: {skipped[:3]}")
    return added, touched


pl_model = YOLO(str(START_WEIGHTS))
for d in prepared_dirs:
    covered = COVERED.get(d.name)
    if covered is None:
        continue
    missing = [CANON_IDX[c] for c in CANONICAL_CLASSES if c not in covered]
    print(f"\n=== {d.name}: размечено {covered}")
    added, touched = add_pseudo_labels(pl_model, d, missing)
    print(f"  картинок дополнено: {touched}; добавлено боксов: "
          f"{ {CANONICAL_CLASSES[k]: v for k, v in sorted(added.items())} }")

# после правки лейблов старые .cache Ultralytics недействительны - удаляем, чтобы не осталось мусора
for c in list(MOCS_YOLO_DIR.rglob("*.cache")) + list(WORK_ROOT.rglob("*.cache")):
    c.unlink()
del pl_model


=== extra_xyzyxzzxy: размечено ['bulldozer', 'concrete_mixer', 'excavator', 'loader', 'other_vehicle', 'roller', 'truck', 'vehicle_crane']
    ...4/17984   ОЗУ процесса: 1.9 ГБ
    ...404/17984   ОЗУ процесса: 2.0 ГБ
    ...804/17984   ОЗУ процесса: 2.0 ГБ
    ...1204/17984   ОЗУ процесса: 2.0 ГБ
    ...1604/17984   ОЗУ процесса: 2.0 ГБ
    ...2004/17984   ОЗУ процесса: 2.0 ГБ
    ...2404/17984   ОЗУ процесса: 2.0 ГБ
    ...2804/17984   ОЗУ процесса: 2.0 ГБ
    ...3204/17984   ОЗУ процесса: 2.0 ГБ
    ...3604/17984   ОЗУ процесса: 2.0 ГБ
    ...4004/17984   ОЗУ процесса: 2.0 ГБ
    ...4404/17984   ОЗУ процесса: 2.0 ГБ
    ...4804/17984   ОЗУ процесса: 2.0 ГБ
    ...5204/17984   ОЗУ процесса: 2.0 ГБ
    ...5604/17984   ОЗУ процесса: 2.0 ГБ
    ...6004/17984   ОЗУ процесса: 2.0 ГБ
    ...6404/17984   ОЗУ процесса: 2.0 ГБ
    ...6804/17984   ОЗУ процесса: 2.0 ГБ
    ...7204/17984   ОЗУ процесса: 2.0 ГБ
    ...7604/17984   ОЗУ процесса: 2.0 ГБ
    ...8004/17984   ОЗУ процесса: 2.0 ГБ
    

> **Проверьте глазами**: откройте 10–20 картинок из `extra_*/images/train` вместе с дописанными боксами
> (например, через `show_frame_with_detections`-подобную отрисовку) — если псевдо-боксы явно мусорные,
> поднимите `PSEUDO_CONF` до 0.7–0.75 или пропустите этот шаг для конкретного датасета.

## 10. Обучающий список: подвыборка MOCS + oversampling редких классов

**Зачем.**
1. MOCS ≈ 37 тыс. train-картинок. На Colab (1–2 vCPU, T4) одна такая эпоха с mosaic-аугментациями идёт очень долго
   и в лимит сессии не влезет. Параметр `fraction` тут не подходит: он берёт **первые N% отсортированного списка путей**, а не случайные —
   в смеси нескольких папок это отрезало бы целые датасеты.
2. Редкие классы (`pile_driver`, `pump_truck`, `concrete_mixer`, `hanging_hook`…) видны модели в единицах процентов кадров.
   Используем *repeat factor sampling* (LVIS): картинка с редким классом повторяется в списке `r = max(1, sqrt(T / f))` раз, где `f` — доля картинок с этим классом.

Из MOCS всегда берутся **все** картинки с редкими классами, остаток добирается случайными до `MOCS_BUDGET`. Доп. датасеты берутся целиком.
Val — по-прежнему все датасеты целиком (по папкам).

In [17]:
import math, random


def build_train_list(datasets, budget=None, rfs_t=RFS_T, max_repeat=MAX_REPEAT, seed=0, out_txt=None):
    rng = random.Random(seed)
    entries = []                                          # (dataset, path, {classes})
    for name, d in datasets.items():
        img_dir, lbl_dir = d / "images/train", d / "labels/train"
        for p in sorted(img_dir.glob("*")):
            if p.suffix.lower() not in IMG_EXT:
                continue
            cs, lp = set(), lbl_dir / (p.stem + ".txt")
            if lp.exists():
                cs = {int(l.split()[0]) for l in lp.read_text().splitlines() if l.strip()}
            entries.append((name, str(p), cs))            # путь НЕ резолвим: Ultralytics выводит labels/ из /images/

    n = len(entries)
    img_freq = Counter(c for _, _, cs in entries for c in cs)
    rc = {c: max(1.0, math.sqrt(rfs_t / (img_freq[c] / n))) for c in img_freq}
    r_of = lambda cs: max([rc[c] for c in cs], default=1.0)

    if budget:
        mocs = [e for e in entries if e[0] == "mocs"]
        others = [e for e in entries if e[0] != "mocs"]
        rare = [e for e in mocs if r_of(e[2]) > 1.0]
        common = [e for e in mocs if r_of(e[2]) <= 1.0]
        rng.shuffle(common)
        entries = others + rare + common[:max(0, budget - len(rare))]

    lines, cls_of = [], {}
    for name, path, cs in entries:
        r = min(max_repeat, r_of(cs))
        k = int(r) + (1 if rng.random() < r - int(r) else 0)
        lines += [path] * max(1, k)
        cls_of[path] = cs
    rng.shuffle(lines)

    if out_txt:
        Path(out_txt).write_text("\n".join(lines))

    before = Counter(c for _, _, cs in entries for c in cs)
    after = Counter(c for pth in lines for c in cls_of[pth])
    stat = pd.DataFrame({
        "img_freq_%_pool": {CANONICAL_CLASSES[c]: round(100 * v / n, 2) for c, v in img_freq.items()},
        "images_in_subset": {CANONICAL_CLASSES[c]: v for c, v in before.items()},
        "images_after_repeat": {CANONICAL_CLASSES[c]: v for c, v in after.items()},
    }).reindex(CANONICAL_CLASSES)
    print(f"пул: {n} картинок -> в список: {len(lines)} (уникальных: {len(entries)})")
    return stat


TRAIN_LIST = WORK_ROOT / "train_stage3.txt"
stat = build_train_list(DATASETS, budget=MOCS_BUDGET, out_txt=TRAIN_LIST)
display(stat)

val_paths = [str((d / "images/val").resolve()) for d in DATASETS.values()
             if (d / "images/val").exists() and any((d / "images/val").iterdir())]
DATA_STAGE3 = WORK_ROOT / "data_stage3.yaml"
DATA_STAGE3.write_text(yaml.safe_dump({
    "train": str(TRAIN_LIST),
    "val": val_paths,
    "names": {i: n for i, n in enumerate(CANONICAL_CLASSES)},
}, allow_unicode=True))
print(DATA_STAGE3.read_text())

пул: 37969 картинок -> в список: 35011 (уникальных: 30565)


,img_freq_%_pool,images_in_subset,images_after_repeat
worker,41.34,9532,12406
tower_crane,10.72,3091,4313
hanging_hook,5.54,2103,3156
vehicle_crane,12.22,4258,5139
roller,1.93,733,1664
bulldozer,2.23,845,1845
excavator,34.58,10614,12362
truck,25.63,8714,9676
loader,11.32,3981,4166
pump_truck,3.46,1314,2251


names:
  0: worker
  1: tower_crane
  2: hanging_hook
  3: vehicle_crane
  4: roller
  5: bulldozer
  6: excavator
  7: truck
  8: loader
  9: pump_truck
  10: concrete_mixer
  11: pile_driver
  12: other_vehicle
train: /content/finetune_data/train_stage3.txt
val:
- /content/mocs_yolo/images/val
- /content/finetune_data/extra_xyzyxzzxy/images/val
- /content/finetune_data/extra_kartaviychert/images/val
- /content/finetune_data/extra_dataclusterlabs/images/val



## 11. Stage 3: дообучение

Отличия от фазы 2 исходного ноутбука (модель уже прошла 50 эпох MOCS + 2 фазы fine-tune):
- **`lr0` 3e-4** (было 5e-4) и `cos_lr` до 5% от старта — не «сломать» уже выученное;
- **`close_mosaic=4`** — последние 4 эпохи без mosaic, на «обычных» кадрах, ближе к реальному видео с камер;
- `patience=6` — стоп, если val не растёт;
- `workers=2`, `cache=False` — под ресурсы Colab;
- `project` — **абсолютный путь на Drive** (относительный Ultralytics кладёт в `runs/detect/...`, из-за этого в исходном ноутбуке путь к `results.csv` не совпадал).

Если сессия оборвалась: заново выполните разделы 0–10 (данные и `train_stage3.txt` должны существовать по тем же путям) и запустите ячейку «Возобновить».

In [ ]:
model = YOLO(str(START_WEIGHTS))       # веса как стартовая точка (не resume)

results = model.train(
    data=str(DATA_STAGE3),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    optimizer="AdamW",
    lr0=3e-4,
    lrf=0.05,
    cos_lr=True,
    warmup_epochs=1,
    close_mosaic=4,
    patience=6,
    cache=False,
    amp=True,
    seed=0,
    project=str(RUNS_DIR),             # абсолютный путь на Drive
    name="stage3",
    exist_ok=True,
)

results_dir = Path(results.save_dir)
best_weights_path = str(results_dir / "weights" / "best.pt")
print("Итоговые лучшие веса:", best_weights_path)

Ultralytics 8.4.157 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=4, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/finetune_data/data_stage3.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.05, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Colab Notebooks/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0,

KeyboardInterrupt: 

In [18]:
# === Возобновить оборванное обучение (запускать ВМЕСТО предыдущей ячейки) ===
model = YOLO(str(RUNS_DIR / "stage3" / "weights" / "last.pt"))
results = model.train(resume=True)
results_dir = Path(results.save_dir); best_weights_path = str(results_dir / "weights" / "best.pt")

Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=4, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/finetune_data/data_stage3.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0003, lrf=0.05, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Colab Notebooks/runs_finetune/stage3/weights/last.pt, momentum=0.937, mosaic=

## 12. Сравнение «до/после»

Смотрите не на общий mAP, а на `Δ AP50-95` по проблемным классам и на датасеты, где раньше было плохо.
Если `stage3` лучше в сумме, но просел на MOCS-классах (`worker`, `tower_crane`, `hanging_hook`) — увеличьте `MOCS_BUDGET`
или уменьшите `lr0`.

In [19]:
after_df = eval_per_dataset(best_weights_path)
cmp = compare_tables(before=baseline_df, after=after_df)
display(cmp)
cmp.to_csv(RUNS_DIR / "stage3_vs_baseline.csv")

Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,847,287 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2216.9±583.7 MB/s, size: 347.2 KB)
val: Scanning /content/mocs_yolo/labels/val... 4000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4000/4000 1.3Kit/s 3.2s
val: New cache created: /content/mocs_yolo/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 250/250 3.1it/s 1:22
                   all       4000      18965      0.852      0.702      0.776      0.599
Speed: 0.9ms preprocess, 15.8ms inference, 0.0ms loss, 1.2ms postprocess per image
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 92 layers, 25,847,287 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1275.9±130.1 MB/s, size: 1246.5 KB)
val: Sca

AP50 [before]  AP50-95 [before]  AP50 [after]  AP50-95 [after]  Δ AP50-95
dataset               class                                                                                    
extra_dataclusterlabs other_vehicle           0.739             0.635         0.799            0.728      0.093
                      truck                   0.860             0.747         0.821            0.741     -0.006
extra_kartaviychert   excavator               0.938             0.567         0.965            0.608      0.041
                      other_vehicle           0.821             0.364         0.872            0.439      0.076
                      truck                   0.801             0.429         0.948            0.502      0.073
                      vehicle_crane           0.930             0.602         0.956            0.681      0.079
extra_xyzyxzzxy       concrete_mixer          0.940             0.774         0.983            0.851      0.077
                      excavator               0.993             0.856         0.994            0.888      0.031
                      loader                  0.969             0.845         0.980            0.881      0.035
                      other_vehicle           0.928             0.754         0.963            0.821      0.067
                      roller                  0.435             0.314         0.745            0.620      0.306
                      truck                   0.985             0.848         0.990            0.889      0.041
                      vehicle_crane           0.987             0.903         0.995            0.944      0.041
mocs                  bulldozer               0.853             0.673         0.877            0.709      0.036
                      concrete_mixer          0.803             0.650         0.804            0.692      0.042
                      excavator               0.924             0.768         0.926            0.777      0.009
                      hanging_hook            0.597             0.327         0.622            0.333      0.006
                      loader                  0.721             0.597         0.749            0.627      0.030
                      other_vehicle           0.578             0.400         0.590            0.415      0.015
                      pile_driver             0.669             0.429         0.652            0.460      0.032
                      pump_truck              0.884             0.688         0.864            0.693      0.004
                      roller                  0.872             0.744         0.879            0.760      0.016
                      tower_crane             0.791             0.592         0.806            0.615      0.023
                      truck                   0.737             0.565         0.746            0.574      0.009
                      vehicle_crane           0.717             0.530         0.720            0.546      0.016
                      worker                  0.846             0.581         0.847            0.584      0.003

## 13. Инференс и трекинг на видео

In [20]:
from datetime import datetime, timedelta
import cv2
import numpy as np

trained_model = YOLO(best_weights_path)


def run_video_tracking(video_path, model, video_start_time, conf=0.35, tracker="bytetrack.yaml",
                        sample_every_n_frames=1, motion_patch_size=48):
    """Детекция + трекинг по видео. Для каждой детекции дополнительно считает motion_energy -
    меру внутрикадрового движения ВНУТРИ рамки объекта (сравнение уменьшенного grayscale-патча
    текущего бокса с патчем того же track_id в предыдущем сохранённом кадре). Это нужно, чтобы
    отличать "работает на месте" (ковш/стрела движутся, а база стоит) от простоя - одно смещение
    центра рамки такое не ловит."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    cap.release()

    prev_patches = {}  # track_id -> последний grayscale-патч
    detections = []

    stream = model.track(source=video_path, conf=conf, tracker=tracker, stream=True, verbose=False)

    for frame_idx, result in enumerate(stream):
        if frame_idx % sample_every_n_frames != 0:
            continue
        timestamp = video_start_time + timedelta(seconds=frame_idx / fps)
        if result.boxes is None:
            continue

        frame = result.orig_img  # BGR numpy-массив текущего кадра, отдаёт сама Ultralytics
        for box in result.boxes:
            class_name = model.names[int(box.cls[0])]
            track_id = int(box.id[0]) if box.id is not None else None
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            motion_energy = None
            if track_id is not None:
                xi1, yi1, xi2, yi2 = int(max(x1, 0)), int(max(y1, 0)), int(x2), int(y2)
                crop = frame[yi1:yi2, xi1:xi2]
                if crop.size > 0:
                    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
                    patch = cv2.resize(gray, (motion_patch_size, motion_patch_size))
                    if track_id in prev_patches:
                        diff = np.abs(patch.astype(np.int16) - prev_patches[track_id].astype(np.int16))
                        motion_energy = float(diff.mean())
                    prev_patches[track_id] = patch

            detections.append({
                "timestamp": timestamp,
                "frame_idx": frame_idx,
                "track_id": track_id,
                "class_name": class_name,
                "confidence": float(box.conf[0]),
                "bbox": (x1, y1, x2, y2),
                "motion_energy": motion_energy,  # None на первом появлении track_id, дальше - число
            })

    return detections


VIDEO_PATH = str(VIDEO_DIR / "your_video.mp4")   # положите видео на Drive или загрузите в /content
detections = run_video_tracking(VIDEO_PATH, trained_model, datetime.now())
print("Всего детекций:", len(detections))

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 2 packages in 242ms
Prepared 1 package in 55ms
Installed 1 package in 1ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 1.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



FileNotFoundError: /content/drive/MyDrive/Colab Notebooks/videos/your_video.mp4 does not exist

### 13b. Экспорт копии видео с отрисованными рамками

Прогоняет видео через `model.track()` ещё раз и для каждого кадра рисует рамки, класс,
track_id и confidence через встроенный `result.plot()`, после чего пишет кадр в выходной
видеофайл через `cv2.VideoWriter`. Результат — mp4-копия исходного видео с наложенной
разметкой в реальном времени (как её "видела" бы модель на каждом кадре).

**Colab:** `mp4v` не играет в браузере. Перекодируйте: `!ffmpeg -y -i your_video_annotated.mp4 -vcodec libx264 annotated_h264.mp4`.

In [ ]:
def export_annotated_video(video_path, model, output_path, conf=0.35, tracker="bytetrack.yaml",
                            progress_every=100, font_size=10, line_width=5):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    # mp4v совместим с большинством плееров (VLC и т.п.); если видео не откроется в браузере,
    # замените fourcc на "avc1" (нужен ffmpeg с поддержкой H.264 в сборке OpenCV).
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (width, height))

    stream = model.track(source=video_path, conf=conf, tracker=tracker, stream=True, verbose=False)
    for frame_idx, result in enumerate(stream):
        annotated_frame = result.plot(font_size=font_size, line_width=line_width)  # BGR-кадр с рамками/классами/id/confidence
        writer.write(annotated_frame)
        if progress_every and frame_idx % progress_every == 0:
            print(f"кадр {frame_idx}/{total_frames}")

    writer.release()
    print("Готово:", output_path)


OUTPUT_VIDEO_PATH = "your_video_annotated.mp4"
export_annotated_video(VIDEO_PATH, trained_model, OUTPUT_VIDEO_PATH)

### 13c. Подсчёт уникальной техники по track_id

In [ ]:
import pandas as pd
from collections import defaultdict


def count_equipment(detections):
    tracks_by_class = defaultdict(set)
    for det in detections:
        if det["track_id"] is not None:
            tracks_by_class[det["class_name"]].add(det["track_id"])
    return {cls: len(ids) for cls, ids in tracks_by_class.items()}


equipment_counts = count_equipment(detections)
for cls, count in equipment_counts.items():
    print(f"  {cls}: {count}")

pd.DataFrame(equipment_counts.items(), columns=["class_name", "count"])

### 13d. Активность и простой техники

Одного смещения центра рамки недостаточно: экскаватор или кран может "работать", стоя на месте
(двигается ковш/стрела), а не только переезжая. Поэтому активность считается по ДВУМ сигналам:
смещение центра рамки (переехала) ИЛИ высокая motion_energy внутри рамки (работает на месте).
Активным считается временное окно, где сработал хотя бы один из сигналов.

In [ ]:
def compute_activity(detections, movement_threshold_px=15, motion_energy_threshold=6.0, window_seconds=60):
    """Группирует детекции по track_id и временным окнам, классифицирует каждое окно
    как "active" или "idle". movement_threshold_px подберите под реальное разрешение видео и
    масштаб сцены (крупный план vs общий вид площадки будут давать разный масштаб смещения
    в пикселях на одно и то же физическое перемещение) - начните с дефолта и скорректируйте
    по факту, глядя на пару размеченных примеров."""
    from collections import defaultdict

    by_track = defaultdict(list)
    for d in detections:
        if d["track_id"] is None:
            continue
        x1, y1, x2, y2 = d["bbox"]
        by_track[d["track_id"]].append({**d, "cx": (x1 + x2) / 2, "cy": (y1 + y2) / 2})

    activity_log = []
    for track_id, points in by_track.items():
        points.sort(key=lambda p: p["timestamp"])
        class_name = points[0]["class_name"]

        window_start_idx = 0
        for i in range(1, len(points)):
            elapsed = (points[i]["timestamp"] - points[window_start_idx]["timestamp"]).total_seconds()
            is_last = i == len(points) - 1
            if elapsed >= window_seconds or is_last:
                seg = points[window_start_idx:i + 1]
                dx = seg[-1]["cx"] - seg[0]["cx"]
                dy = seg[-1]["cy"] - seg[0]["cy"]
                displacement = (dx ** 2 + dy ** 2) ** 0.5

                motion_values = [p["motion_energy"] for p in seg if p["motion_energy"] is not None]
                max_motion = max(motion_values) if motion_values else 0.0

                moved = displacement > movement_threshold_px
                worked_in_place = max_motion > motion_energy_threshold
                status = "active" if (moved or worked_in_place) else "idle"

                activity_log.append({
                    "track_id": track_id, "class_name": class_name,
                    "window_start": seg[0]["timestamp"], "window_end": seg[-1]["timestamp"],
                    "displacement_px": round(displacement, 1), "max_motion_energy": round(max_motion, 2),
                    "status": status,
                    "reason": "moved" if moved else ("worked_in_place" if worked_in_place else "idle"),
                })
                window_start_idx = i
    return activity_log


def summarize_activity(activity_log):
    """Сводка по каждой единице техники: сколько окон активна/простаивает, доля простоя -
    это то, что пойдёт в отчёт как основной вывод модуля."""
    from collections import defaultdict

    stats_by_track = defaultdict(lambda: {"active": 0, "idle": 0, "class_name": None})
    for entry in activity_log:
        s = stats_by_track[entry["track_id"]]
        s["class_name"] = entry["class_name"]
        s[entry["status"]] += 1

    rows = []
    for track_id, s in stats_by_track.items():
        total = s["active"] + s["idle"]
        rows.append({
            "track_id": track_id, "class_name": s["class_name"],
            "active_windows": s["active"], "idle_windows": s["idle"],
            "idle_share": round(s["idle"] / total, 2) if total else 0,
        })
    return pd.DataFrame(rows).sort_values("idle_share", ascending=False)


activity_log = compute_activity(detections, window_seconds=60)
activity_summary = summarize_activity(activity_log)
activity_summary

### 13e. События прибытия/исчезновения техники

Строится журнал событий: для каждого отслеживаемого объекта (`track_id`) берётся
первое появление в кадре (`arrival`) и последнее (`departure`). Класс техники берётся
как самый частый `class_name` среди детекций этого объекта.

Трекер (ByteTrack) может потерять объект при долгом перекрытии и выдать ему при повторном
появлении новый `track_id` — тогда вместо одного объекта получились бы "лишние"
departure + arrival. `merge_track_splits` склеивает такие разорванные треки: если трек
того же класса начинается вскоре после (`max_gap_seconds`) и рядом в кадре
(`max_center_dist_px`) с концом другого, они считаются одним объектом.

In [ ]:
from typing import Literal
from pydantic import BaseModel


# Классы объектов детектора MOCS (Moving Objects on Construction Sites).
EquipmentClass = Literal[
    "worker",
    "tower_crane",
    "hanging_hook",
    "vehicle_crane",
    "roller",
    "bulldozer",
    "excavator",
    "truck",
    "loader",
    "pump_truck",
    "concrete_mixer",
    "pile_driver",
    "other_vehicle",
]


class EquipmentEvent(BaseModel):
    track_id: int
    equipment_class: EquipmentClass
    event: Literal["arrival", "departure"]
    at: datetime


def _bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def _dist(p1, p2):
    return ((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2) ** 0.5


def _build_track_summaries(df):
    summaries = []
    for track_id, group in df.groupby("track_id"):
        group = group.sort_values("timestamp")
        first, last = group.iloc[0], group.iloc[-1]
        summaries.append({
            "track_id": int(track_id),
            "equipment_class": group["class_name"].mode().iloc[0],
            "start_time": first["timestamp"],
            "end_time": last["timestamp"],
            "first_bbox_center": _bbox_center(first["bbox"]),
            "last_bbox_center": _bbox_center(last["bbox"]),
        })
    return summaries


def merge_track_splits(track_summaries, max_gap_seconds=3.0, max_center_dist_px=80.0):
    """Объединяет разорванные треки: если байт-трекер после долгого перекрытия выдал
    объекту новый track_id, то трек того же класса, начавшийся вскоре (<= max_gap_seconds)
    и рядом в кадре (<= max_center_dist_px от последнего bbox) с концом предыдущего трека,
    считается продолжением того же объекта. Возвращает {track_id: id_объединённой_группы}."""
    parent = {t["track_id"]: t["track_id"] for t in track_summaries}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    ordered = sorted(track_summaries, key=lambda t: t["start_time"])
    open_chains = []  # хвосты ещё не завершённых цепочек треков

    for track in ordered:
        best_chain, best_gap = None, None
        for chain in open_chains:
            if chain["equipment_class"] != track["equipment_class"]:
                continue
            gap = (track["start_time"] - chain["end_time"]).total_seconds()
            if gap < 0 or gap > max_gap_seconds:
                continue
            if _dist(chain["last_bbox_center"], track["first_bbox_center"]) > max_center_dist_px:
                continue
            if best_gap is None or gap < best_gap:
                best_gap, best_chain = gap, chain

        if best_chain is not None:
            union(best_chain["track_id"], track["track_id"])
            open_chains.remove(best_chain)

        open_chains.append({
            "track_id": track["track_id"],
            "end_time": track["end_time"],
            "last_bbox_center": track["last_bbox_center"],
            "equipment_class": track["equipment_class"],
        })

    return {t["track_id"]: find(t["track_id"]) for t in track_summaries}


def build_equipment_events(detections, max_gap_seconds=3.0, max_center_dist_px=80.0):
    """Строит список событий arrival/departure по объектам, предварительно склеив разорванные
    треки через merge_track_splits. track_id события - наименьший исходный track_id в группе."""
    if not detections:
        return []

    df = pd.DataFrame(detections)
    df = df[df["track_id"].notna()].copy()
    df["track_id"] = df["track_id"].astype(int)

    track_summaries = _build_track_summaries(df)
    merge_map = merge_track_splits(
        track_summaries, max_gap_seconds=max_gap_seconds, max_center_dist_px=max_center_dist_px,
    )
    df["group_id"] = df["track_id"].map(merge_map)

    events = []
    for group_id, group in df.groupby("group_id"):
        equipment_class = group["class_name"].mode().iloc[0]
        representative_track_id = int(group["track_id"].min())
        events.append(EquipmentEvent(
            track_id=representative_track_id, equipment_class=equipment_class,
            event="arrival", at=group["timestamp"].min(),
        ))
        events.append(EquipmentEvent(
            track_id=representative_track_id, equipment_class=equipment_class,
            event="departure", at=group["timestamp"].max(),
        ))

    events.sort(key=lambda e: e.at)
    return events


def format_elapsed(at, start_time):
    total_seconds = int((at - start_time).total_seconds())
    minutes, seconds = divmod(total_seconds, 60)
    return f"{minutes:02d}:{seconds:02d}"


equipment_events = build_equipment_events(detections)
video_start = min(d["timestamp"] for d in detections) if detections else None

for ev in equipment_events:
    verb = "прибыл" if ev.event == "arrival" else "исчез"
    print(f"{ev.equipment_class} #{ev.track_id} {verb} на {format_elapsed(ev.at, video_start)}")

events_path = "equipment_events.json"
with open(events_path, "w", encoding="utf-8") as f:
    json.dump([e.model_dump(mode="json") for e in equipment_events], f, ensure_ascii=False, indent=2)
print("Сохранено:", events_path)

### 13f. Визуализация детекций на кадре

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def show_frame_with_detections(video_path, frame_idx, detections):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        print("Не удалось прочитать кадр", frame_idx)
        return

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.imshow(frame_rgb)

    for det in [d for d in detections if d["frame_idx"] == frame_idx]:
        x1, y1, x2, y2 = det["bbox"]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="red", facecolor="none")
        ax.add_patch(rect)
        ax.text(x1, y1 - 5, f"{det['class_name']} #{det['track_id']}", color="red", fontsize=10, weight="bold")

    ax.axis("off")
    plt.show()


if detections:
    mid_frame = detections[len(detections) // 2]["frame_idx"]
    show_frame_with_detections(VIDEO_PATH, mid_frame, detections)

## 14. Графики динамики обучения (stage 3)

In [ ]:
csv_path = results_dir / "results.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Fine-tune (stage 3) — динамика обучения", fontsize=16, fontweight="bold")

epoch = df["epoch"] + 1

axes[0, 0].plot(epoch, df["train/box_loss"], label="box_loss")
axes[0, 0].plot(epoch, df["train/cls_loss"], label="cls_loss")
axes[0, 0].plot(epoch, df["train/dfl_loss"], label="dfl_loss")
axes[0, 0].set_title("Ошибки на train")
axes[0, 0].set_xlabel("Эпоха"); axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(alpha=0.3); axes[0, 0].legend()

for col in ["val/box_loss", "val/cls_loss", "val/dfl_loss"]:
    if col in df.columns:
        axes[0, 1].plot(epoch, df[col], label=col.replace("val/", ""))
axes[0, 1].set_title("Ошибки на validation")
axes[0, 1].set_xlabel("Эпоха"); axes[0, 1].set_ylabel("Loss")
axes[0, 1].grid(alpha=0.3); axes[0, 1].legend()

axes[1, 0].plot(epoch, df["metrics/precision(B)"], label="Precision")
axes[1, 0].plot(epoch, df["metrics/recall(B)"], label="Recall")
axes[1, 0].set_title("Precision и Recall")
axes[1, 0].set_xlabel("Эпоха"); axes[1, 0].set_ylabel("Значение")
axes[1, 0].set_ylim(0, 1); axes[1, 0].grid(alpha=0.3); axes[1, 0].legend()

axes[1, 1].plot(epoch, df["metrics/mAP50(B)"], label="mAP50")
axes[1, 1].plot(epoch, df["metrics/mAP50-95(B)"], label="mAP50-95")
axes[1, 1].set_title("Качество детекции")
axes[1, 1].set_xlabel("Эпоха"); axes[1, 1].set_ylabel("mAP")
axes[1, 1].set_ylim(0, 1); axes[1, 1].grid(alpha=0.3); axes[1, 1].legend()

plt.tight_layout()
output_path = csv_path.parent / "metrics_over_epochs.png"
plt.savefig(output_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"График сохранён: {output_path}")